# PicoCal — Minimum bias vs no minimum bias (notebook 09)

New data in `data/gsoc_drive/`: `with_minimum_bias` vs `without_minimum_bias`. Different file ranges, so this is a statistical comparison, not per-event.

## Schema and size check

In [1]:
import sys
from pathlib import Path
import numpy as np
import awkward as ak
import uproot

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import resolution

BR = ["energy", "total_energy", "sig_flux_eTot", "sig_flux_prod_vertex_z"]

def load(folder):
    files = sorted((repo / "data" / "gsoc_drive" / folder).glob("matched_*.root"))
    E, TE, SUM, NC = [], [], [], []
    for path in files:
        with uproot.open(path) as f:
            a = f["clusters_matched"].arrays(BR, library="ak")
        vz = ak.to_numpy(a["sig_flux_prod_vertex_z"]).astype(float)
        keep = vz < 100.0
        e = a["energy"][keep]
        E.append(ak.to_numpy(a["sig_flux_eTot"][keep]).astype(float))
        TE.append(ak.to_numpy(a["total_energy"][keep]).astype(float))
        SUM.append(ak.to_numpy(ak.sum(e, axis=1)).astype(float))
        NC.append(ak.to_numpy(ak.num(e)).astype(float))
    return {"n_files": len(files), "Etrue": np.concatenate(E), "total_energy": np.concatenate(TE),
            "sumE": np.concatenate(SUM), "ncells": np.concatenate(NC)}

without_mb = load("without_minimum_bias")
with_mb = load("with_minimum_bias")

full_file = sorted((repo / "data" / "full").glob("matched_*.root"))[0]
with uproot.open(full_file) as f:
    nfull = ak.to_numpy(ak.num(f["clusters_matched"]["energy"].array()))

{"without_pileup_clusters": len(without_mb["Etrue"]),
 "with_pileup_clusters": len(with_mb["Etrue"]),
 "median_cells_without": float(np.median(without_mb["ncells"])),
 "median_cells_with": float(np.median(with_mb["ncells"])),
 "median_cells_data_full": float(np.median(nfull))}

{'without_pileup_clusters': 1837,
 'with_pileup_clusters': 893,
 'median_cells_without': 9.0,
 'median_cells_with': 9.0,
 'median_cells_data_full': 132.0}

## Distributions: with vs without pileup

In [2]:
import plotly.graph_objects as go

C_WITHOUT, C_WITH = "#4c78a8", "#e45756"

def overlay(getter, bins, title, xlab, xrange=None):
    fig = go.Figure()
    for name, D, col in [("without pileup", without_mb, C_WITHOUT), ("with pileup", with_mb, C_WITH)]:
        h, edges = np.histogram(getter(D), bins=bins, density=True)
        ctr = (edges[:-1] + edges[1:]) / 2
        fig.add_trace(go.Bar(x=ctr, y=h, name=name, marker_color=col, opacity=0.6))
    fig.update_layout(barmode="overlay", template="plotly_white", height=380, title=title,
                      xaxis_title=xlab, yaxis_title="density", legend_title="")
    if xrange:
        fig.update_xaxes(range=xrange)
    return fig

overlay(lambda D: D["ncells"], np.arange(0.5, 13.5, 1), "Cells per cluster", "cells").show()
overlay(lambda D: D["Etrue"], np.linspace(0, 60, 40), "True photon energy (sig_flux_eTot)", "E_true").show()
overlay(lambda D: D["total_energy"] / 1000.0, np.linspace(0, 60, 40), "Cluster total_energy (/1000)", "total_energy/1000").show()

## Does pileup degrade the naive energy estimate?

In [3]:
import pandas as pd
EPS = 1e-6

def calib_pred(x, E):
    m = (x > 0) & (E > 0)
    a, b = np.polyfit(np.log(x[m] + EPS), np.log(E[m] + EPS), 1)
    return np.exp(a * np.log(x[m] + EPS) + b), E[m]

rows = []
for name, D in [("without_pileup", without_mb), ("with_pileup", with_mb)]:
    pt, et = calib_pred(D["total_energy"], D["Etrue"])
    ps, es = calib_pred(D["sumE"], D["Etrue"])
    rows.append({"dataset": name, "n": len(D["Etrue"]), "median_cells": float(np.median(D["ncells"])),
                 "sigma_total_energy": resolution(pt, et)["sigma_eff"],
                 "sigma_sum": resolution(ps, es)["sigma_eff"]})
res_table = pd.DataFrame(rows)
res_table

,dataset,n,median_cells,sigma_total_energy,sigma_sum
0,without_pileup,1837,9.0,0.4760,0.4760
1,with_pileup,893,9.0,0.1207,0.1207


In [4]:
fig = go.Figure()
for name, D, col in [("without pileup", without_mb, C_WITHOUT), ("with pileup", with_mb, C_WITH)]:
    pt, et = calib_pred(D["total_energy"], D["Etrue"])
    r = (pt - et) / et
    h, edges = np.histogram(r, bins=np.linspace(-0.6, 0.6, 60), density=True)
    ctr = (edges[:-1] + edges[1:]) / 2
    fig.add_trace(go.Bar(x=ctr, y=h, name=name, marker_color=col, opacity=0.6))
fig.update_layout(barmode="overlay", template="plotly_white", height=400,
                  title="Calibrated total_energy resolution r = (E_pred - E_true)/E_true",
                  xaxis_title="r", yaxis_title="density", legend_title="")
fig.show()

### Energy-matched check (10-40 GeV)

In [5]:
rows = []
for name, D in [("without_pileup", without_mb), ("with_pileup", with_mb)]:
    E = D["Etrue"]; x = D["total_energy"]
    m = (E > 10) & (E < 40) & (x > 0)
    a, b = np.polyfit(np.log(x[m]), np.log(E[m]), 1)
    pred = np.exp(a * np.log(x[m]) + b)
    rows.append({"dataset": name, "n_10_40GeV": int(m.sum()),
                 "Etrue_median": round(float(np.median(E)), 1),
                 "sigma_matched": resolution(pred, E[m])["sigma_eff"]})
pd.DataFrame(rows)

,dataset,n_10_40GeV,Etrue_median,sigma_matched
0,without_pileup,939,27.6,0.3139
1,with_pileup,433,27.4,0.1150


**Surprising result, reported honestly.** With-pileup is *not* worse here - it is better (sigma 0.12 vs 0.48 overall, 0.12 vs 0.31 energy-matched), the opposite of the naive pileup-degrades expectation. The spectra are similar (both median ~27 GeV), so it is not an energy-spectrum artifact. The most likely cause is **selection**: pileup matching keeps only ~half the clusters (893 vs 1837), probably the cleaner, well-contained ones. So this simple with-vs-without comparison cannot test pileup degradation on its own - it needs a per-event or selection-controlled study. Open question to raise with the mentors.

### Why: containment of the 9-cell window

In [6]:
C_without = without_mb["total_energy"] / 1000.0 / np.maximum(without_mb["Etrue"], 1e-9)
C_with = with_mb["total_energy"] / 1000.0 / np.maximum(with_mb["Etrue"], 1e-9)

fig = go.Figure()
for name, C, col in [("without pileup", C_without, C_WITHOUT), ("with pileup", C_with, C_WITH)]:
    h, edges = np.histogram(C, bins=np.linspace(0, 2.5, 60), density=True)
    ctr = (edges[:-1] + edges[1:]) / 2
    fig.add_trace(go.Bar(x=ctr, y=h, name=name, marker_color=col, opacity=0.6))
fig.add_vline(x=1.0, line_dash="dash", line_color="#333")
fig.update_layout(barmode="overlay", template="plotly_white", height=400,
                  title="Containment C = total_energy / E_true (9-cell window)",
                  xaxis_title="C", yaxis_title="density", legend_title="")
fig.show()

pd.DataFrame([{"dataset": n, "median_C": round(float(np.median(C)), 3),
               "frac_C<0.5": round(float((C < 0.5).mean()), 3),
               "frac_C>1.2": round(float((C > 1.2).mean()), 3)}
              for n, C in [("without_pileup", C_without), ("with_pileup", C_with)]])

,dataset,median_C,frac_C<0.5,frac_C>1.2
0,without_pileup,0.914,0.105,0.106
1,with_pileup,1.020,0.016,0.188


The naive resolution gap is a **selection artifact**, now explained. Without pileup, ~10% of clusters are badly under-contained (`C < 0.5`) - the 9-cell window misses most of the shower - and that tail wrecks the resolution. The pileup sample's matching keeps only ~half the clusters and removes almost all under-contained ones (1.6% left), so it looks tighter. Pileup *does* add energy (median C 0.91 -> 1.02, more `C > 1.2`), as expected. Two real takeaways: (1) the 9-cell pre-window is too tight without pileup; (2) with/without cannot be compared directly - the matching selection differs. Both are questions for the mentors.